In [ ]:
"""
MAHED Task 1 Gemini prompting experiment.

Experiment:
- Create or load a fixed stratified validation subset of 500 samples.
- Run Gemini 3 Flash
- Use batched inference with 25 samples per request.
- Save subset, row-level predictions, and metrics to Google Drive.
- Resume automatically from the last unprocessed sample if Colab stops.

Before running in Colab:
    !pip install -q -U google-genai scikit-learn pandas

Expected MAHED columns:
    text, label

Expected Colab secret:
    gemini
"""

import os
import re
import time
import json
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

try:
    from google.colab import userdata
except Exception:
    userdata = None

from google import genai


# ============================================================
# Configuration
# ============================================================
SEED = 42
MODEL_NAME = "gemini-3-flash-preview"
BATCH_SIZE = 25
SLEEP_SEC = 12
MAX_RETRIES = 2
SUBSET_SIZE = 500
FEW_SHOTS_PER_CLASS = 1

VALID_LABELS = ["hate", "hope", "not_applicable"]
VALID_LABELS_SET = set(VALID_LABELS)




#### IMPORTANT!!!: Use valid file paths
TRAIN_PATH = "/content/drive/MyDrive/496/MAHED Sub Task 1/MAHED Sub Task 1/train.csv"
VAL_PATH = "/content/drive/MyDrive/496/MAHED Sub Task 1/MAHED Sub Task 1/validation.csv"

# Change this folder if needed.
#### IMPORTANT!!!: Use valid file paths
OUTPUT_DIR = "/content/drive/MyDrive/496/MAHED_Gemini_Task3"
SUBSET_PATH = f"{OUTPUT_DIR}/mahed_val_subset_500.csv"
FEW_SHOT_EXAMPLES_PATH = f"{OUTPUT_DIR}/few_shot_examples.csv"
SUMMARY_METRICS_PATH = f"{OUTPUT_DIR}/metrics_summary.csv"
PER_CLASS_METRICS_PATH = f"{OUTPUT_DIR}/metrics_per_class.csv"
CONFUSION_MATRIX_PATH = f"{OUTPUT_DIR}/confusion_matrices.json"


#### Choose prompt methods:
PROMPT_TYPES = ["three_shot_cultural"]


# ============================================================
# Data utilities
# ============================================================
def ensure_output_dir() -> None:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def clean_label_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower()


def load_mahed_data(train_path: str = TRAIN_PATH, val_path: str = VAL_PATH) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    required_cols = {"text", "label"}
    for name, df in [("train", train_df), ("validation", val_df)]:
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"{name} file is missing columns: {sorted(missing)}")

    train_df = train_df.copy()
    val_df = val_df.copy()

    train_df["text"] = train_df["text"].astype(str)
    val_df["text"] = val_df["text"].astype(str)
    train_df["label"] = clean_label_series(train_df["label"])
    val_df["label"] = clean_label_series(val_df["label"])

    train_df = train_df[train_df["label"].isin(VALID_LABELS)].reset_index(drop=True)
    val_df = val_df[val_df["label"].isin(VALID_LABELS)].reset_index(drop=True)

    print("Train distribution:")
    print(train_df["label"].value_counts())
    print("\nValidation distribution:")
    print(val_df["label"].value_counts())

    return train_df, val_df


def stratified_subset(
    df: pd.DataFrame,
    total_n: int = SUBSET_SIZE,
    label_col: str = "label",
    random_state: int = SEED,
) -> pd.DataFrame:
    """Create a stratified subset that preserves the validation label distribution."""
    if total_n > len(df):
        raise ValueError(f"Requested {total_n} samples, but dataframe has only {len(df)} rows.")

    proportions = df[label_col].value_counts(normalize=True).reindex(VALID_LABELS)
    counts = (proportions * total_n).round().astype(int)

    # Fix rounding drift so the final subset has exactly total_n rows.
    diff = total_n - int(counts.sum())
    if diff != 0:
        largest_class = counts.idxmax()
        counts.loc[largest_class] += diff

    parts = []
    for label in VALID_LABELS:
        class_df = df[df[label_col] == label]
        n = int(counts.loc[label])
        if n > len(class_df):
            raise ValueError(f"Not enough rows for label={label}. Need {n}, available {len(class_df)}.")
        parts.append(class_df.sample(n=n, random_state=random_state))

    subset = pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    subset.insert(0, "sample_id", range(len(subset)))
    return subset


def load_or_create_subset(val_df: pd.DataFrame) -> pd.DataFrame:
    ensure_output_dir()
    if os.path.exists(SUBSET_PATH):
        subset = pd.read_csv(SUBSET_PATH)
        print(f"Loaded existing subset: {SUBSET_PATH}")
    else:
        subset = stratified_subset(val_df, total_n=SUBSET_SIZE)
        subset.to_csv(SUBSET_PATH, index=False)
        print(f"Saved new subset: {SUBSET_PATH}")

    print("\nSubset distribution:")
    print(subset["label"].value_counts())
    return subset


def load_or_create_few_shot_examples(train_df: pd.DataFrame) -> pd.DataFrame:
    ensure_output_dir()
    if os.path.exists(FEW_SHOT_EXAMPLES_PATH):
        examples = pd.read_csv(FEW_SHOT_EXAMPLES_PATH)
        print(f"Loaded existing few-shot examples: {FEW_SHOT_EXAMPLES_PATH}")
        return examples

    parts = []
    for label in VALID_LABELS:
        class_df = train_df[train_df["label"] == label]
        examples = class_df.sample(n=FEW_SHOTS_PER_CLASS, random_state=SEED)
        parts.append(examples[["text", "label"]])

    examples_df = pd.concat(parts).reset_index(drop=True)
    examples_df.to_csv(FEW_SHOT_EXAMPLES_PATH, index=False)
    print(f"Saved few-shot examples: {FEW_SHOT_EXAMPLES_PATH}")
    return examples_df


# ============================================================
# Gemini utilities
# ============================================================
def get_gemini_client() -> genai.Client:
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key and userdata is not None:
        api_key = userdata.get("gemini")

    if not api_key:
        raise RuntimeError(
            "Gemini API key not found. Add a Colab secret named 'gemini' or set GEMINI_API_KEY."
        )

    os.environ["GEMINI_API_KEY"] = api_key
    return genai.Client(api_key=api_key)


def normalize_prediction(pred_text: str) -> str:
    pred = str(pred_text).strip().lower()
    pred = pred.replace(" ", "").replace("\n", "").replace("\t", "")

    mapping = {
        "hate": "hate",
        "hope": "hope",
        "not_applicable": "not_applicable",
        "notapplicable": "not_applicable",
        "not-applicable": "not_applicable",
    }
    return mapping.get(pred, "INVALID")


def format_numbered_texts(texts: List[str]) -> str:
    return "\n".join([f"{i + 1}. {str(text).strip()}" for i, text in enumerate(texts)])


def format_few_shot_examples(examples_df: pd.DataFrame) -> str:
    lines = []
    for _, row in examples_df.iterrows():
        lines.append(f'Text: {str(row["text"]).strip()}')
        lines.append(f'Label: {str(row["label"]).strip()}')
        lines.append("")
    return "\n".join(lines).strip()


def build_batch_prompt(
    texts: List[str],
    prompt_type: str,
    few_shot_examples: pd.DataFrame | None = None,
) -> str:
    numbered_texts = format_numbered_texts(texts)

    definition_labels = """Labels:
hate = hateful, hostile, abusive, or harmful toward a person or group
hope = supportive, encouraging, empathetic, or constructive
not_applicable = neither hate nor hope, or unclear/neutral"""

    nli_labels = """Labels:
hate = the text entails hateful, hostile, abusive, or harmful meaning toward a person or group
hope = the text entails supportive, encouraging, empathetic, or constructive meaning
not_applicable = neither hate nor hope is entailed, or the meaning is unclear/neutral"""

    distinction_labels = """Use these distinctions:
hate = attacks, dehumanizes, insults, threatens, or promotes hostility toward a person or group
hope = expresses support, empathy, encouragement, solidarity, or positive action
toxic_or_offensive_but_not_hate = rude, aggressive, insulting, or offensive, but not clearly hate and not clearly hope
not_applicable = neutral, unclear, irrelevant, or does not fit hate or hope

Important rule:
If the text is only rude, offensive, political criticism, or general anger, do not label it hate unless it clearly targets a person or group with harmful or hateful meaning."""

    cultural_labels = """Labels:
hate = hateful, hostile, abusive, or harmful toward a person or group
hope = supportive, encouraging, empathetic, or constructive
not_applicable = neither hate nor hope, or unclear/neutral

Consider Arabic cultural context, dialectal wording, religious expressions, political language, sarcasm, and indirect meaning. Do not classify a text as hate only because it contains negative words. Use the intended meaning of the full text."""

    output_rule = """Return exactly one line per text using this format:
1: hate
2: hope
3: not_applicable

Use only these labels: hate, hope, not_applicable.
Do not explain."""

    if prompt_type == "zero_shot_definition":
        return f"""Classify each Arabic text into one label only.

{definition_labels}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "zero_shot_nli":
        return f"""Classify each Arabic text using an NLI-style decision.

For each text, decide which label is best entailed by the meaning of the text.

{nli_labels}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "zero_shot_distinction":
        return f"""Classify each Arabic text into one label only.

{distinction_labels}

Final labels allowed:
hate
hope
not_applicable

Map toxic_or_offensive_but_not_hate to not_applicable.

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "few_shot_definition":
        if few_shot_examples is None or few_shot_examples.empty:
            raise ValueError("few_shot_definition requires few_shot_examples.")
        examples = format_few_shot_examples(few_shot_examples)

        return f"""Classify each Arabic text into one label only.

{definition_labels}

Examples:
{examples}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "three_shot_cultural":
        if few_shot_examples is None or few_shot_examples.empty:
            raise ValueError("three_shot_cultural requires few_shot_examples.")
        examples = format_few_shot_examples(few_shot_examples)

        return f"""Classify each Arabic text into one label only.

{cultural_labels}

Examples:
{examples}

{output_rule}

Texts:
{numbered_texts}""".strip()

    raise ValueError(f"Unknown prompt_type: {prompt_type}")

def parse_batch_output(raw_text: str, batch_size: int) -> List[str]:
    preds = ["INVALID"] * batch_size
    lines = str(raw_text).strip().splitlines()

    for line in lines:
        line = line.strip()
        match = re.match(r"^\s*(\d+)\s*[:\-\)]\s*(.+?)\s*$", line)
        if not match:
            continue

        idx = int(match.group(1)) - 1
        label_text = match.group(2)
        if 0 <= idx < batch_size:
            preds[idx] = normalize_prediction(label_text)

    return preds


def predict_gemini_batch(
    client: genai.Client,
    texts: List[str],
    prompt_type: str,
    few_shot_examples: pd.DataFrame | None = None,
    model: str = MODEL_NAME,
    max_retries: int = MAX_RETRIES,
) -> Tuple[str, List[str]]:
    prompt = build_batch_prompt(texts, prompt_type, few_shot_examples)

    for attempt in range(max_retries + 1):
        try:
            response = client.models.generate_content(model=model, contents=prompt)
            raw = response.text
            preds = parse_batch_output(raw, len(texts))
            return raw, preds
        except Exception as exc:
            if attempt == max_retries:
                raise RuntimeError(f"Gemini request failed after {max_retries + 1} attempts: {exc}") from exc
            wait_time = 10 * (attempt + 1)
            print(f"Request failed. Retrying in {wait_time}s. Error: {exc}")
            time.sleep(wait_time)

    raise RuntimeError("Unexpected Gemini failure.")


# ============================================================
# Results and resume utilities
# ============================================================
def results_path_for(prompt_type: str) -> str:
    return f"{OUTPUT_DIR}/results_{prompt_type}_gemini_2_5_flash.csv"


def init_or_load_results(subset_df: pd.DataFrame, prompt_type: str) -> pd.DataFrame:
    path = results_path_for(prompt_type)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded existing results for {prompt_type}: {path}")
    else:
        keep_cols = ["sample_id", "text", "label"]
        df = subset_df[keep_cols].copy()
        df["prompt_type"] = prompt_type
        df["model"] = MODEL_NAME
        df["pred"] = ""
        df["raw_output"] = ""
        df["processed"] = False
        df["correct"] = ""
        df["batch_id"] = ""
        df["timestamp"] = ""
        df.to_csv(path, index=False)
        print(f"Created results file for {prompt_type}: {path}")

    if "processed" in df.columns:
        df["processed"] = df["processed"].astype(str).str.lower().isin(["true", "1", "yes"])
    else:
        df["processed"] = False

    return df


def run_prompt_experiment(
    client: genai.Client,
    subset_df: pd.DataFrame,
    prompt_type: str,
    few_shot_examples: pd.DataFrame | None = None,
    batch_size: int = BATCH_SIZE,
    sleep_sec: int = SLEEP_SEC,
) -> pd.DataFrame:
    path = results_path_for(prompt_type)
    df = init_or_load_results(subset_df, prompt_type)

    remaining_indices = df.index[~df["processed"]].tolist()
    print(f"\nPrompt: {prompt_type}")
    print(f"Remaining samples: {len(remaining_indices)} / {len(df)}")

    if not remaining_indices:
        print("No remaining samples. Skipping inference.")
        return df

    batch_counter = 0
    for start in range(0, len(remaining_indices), batch_size):
        batch_indices = remaining_indices[start:start + batch_size]
        batch_texts = df.loc[batch_indices, "text"].astype(str).tolist()
        batch_id = f"{prompt_type}_batch_{start // batch_size:04d}"

        try:
            raw_output, preds = predict_gemini_batch(
                client=client,
                texts=batch_texts,
                prompt_type=prompt_type,
                few_shot_examples=few_shot_examples,
                model=MODEL_NAME,
            )
        except RuntimeError as exc:
            print(f"Stopping experiment. Current progress was saved. Error: {exc}")
            df.to_csv(path, index=False)
            break

        timestamp = pd.Timestamp.now().isoformat()
        for local_i, row_idx in enumerate(batch_indices):
            pred = preds[local_i] if local_i < len(preds) else "INVALID"
            gold = str(df.loc[row_idx, "label"]).strip().lower()

            df.loc[row_idx, "pred"] = pred
            df.loc[row_idx, "raw_output"] = raw_output
            df.loc[row_idx, "processed"] = True
            df.loc[row_idx, "correct"] = int(pred == gold)
            df.loc[row_idx, "batch_id"] = batch_id
            df.loc[row_idx, "timestamp"] = timestamp

        df.to_csv(path, index=False)
        batch_counter += 1
        processed_total = int(df["processed"].sum())
        print(f"Saved {batch_id}. Processed {processed_total} / {len(df)}")

        if start + batch_size < len(remaining_indices):
            time.sleep(sleep_sec)

    return df


# ============================================================
# Metrics
# ============================================================
def compute_metrics_for_results(results_df: pd.DataFrame, prompt_type: str) -> Tuple[Dict, pd.DataFrame, Dict]:
    df = results_df.copy()
    df["label"] = clean_label_series(df["label"])
    df["pred"] = df["pred"].astype(str).str.strip().str.lower()

    processed_df = df[df["processed"].astype(str).str.lower().isin(["true", "1", "yes"])].copy()
    valid_df = processed_df[
        processed_df["label"].isin(VALID_LABELS) & processed_df["pred"].isin(VALID_LABELS)
    ].copy()

    summary = {
        "prompt_type": prompt_type,
        "model": MODEL_NAME,
        "total_samples": len(df),
        "processed_samples": len(processed_df),
        "valid_predictions": len(valid_df),
        "invalid_predictions": int(len(processed_df) - len(valid_df)),
        "invalid_rate_processed": 0.0 if len(processed_df) == 0 else (len(processed_df) - len(valid_df)) / len(processed_df),
        "accuracy": None,
        "precision_macro": None,
        "recall_macro": None,
        "f1_macro": None,
        "precision_micro": None,
        "recall_micro": None,
        "f1_micro": None,
        "precision_weighted": None,
        "recall_weighted": None,
        "f1_weighted": None,
    }

    if len(valid_df) == 0:
        per_class_df = pd.DataFrame(columns=["prompt_type", "label", "precision", "recall", "f1", "support"])
        return summary, per_class_df, {}

    y_true = valid_df["label"].tolist()
    y_pred = valid_df["pred"].tolist()

    summary.update({
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    })

    report = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        output_dict=True,
        zero_division=0,
    )

    per_class_rows = []
    for label in VALID_LABELS:
        per_class_rows.append({
            "prompt_type": prompt_type,
            "label": label,
            "precision": report[label]["precision"],
            "recall": report[label]["recall"],
            "f1": report[label]["f1-score"],
            "support": report[label]["support"],
        })
    per_class_df = pd.DataFrame(per_class_rows)

    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
    cm_dict = {
        "labels": VALID_LABELS,
        "matrix": cm.tolist(),
    }

    return summary, per_class_df, cm_dict


def save_all_metrics(prompt_types: List[str]) -> None:
    summary_rows = []
    per_class_frames = []
    confusion_matrices = {}

    for prompt_type in prompt_types:
        path = results_path_for(prompt_type)
        if not os.path.exists(path):
            continue

        results_df = pd.read_csv(path)
        summary, per_class_df, cm_dict = compute_metrics_for_results(results_df, prompt_type)
        summary_rows.append(summary)
        per_class_frames.append(per_class_df)
        confusion_matrices[prompt_type] = cm_dict

    summary_df = pd.DataFrame(summary_rows)
    per_class_all_df = pd.concat(per_class_frames, ignore_index=True) if per_class_frames else pd.DataFrame()

    summary_df.to_csv(SUMMARY_METRICS_PATH, index=False)
    per_class_all_df.to_csv(PER_CLASS_METRICS_PATH, index=False)

    with open(CONFUSION_MATRIX_PATH, "w", encoding="utf-8") as f:
        json.dump(confusion_matrices, f, ensure_ascii=False, indent=2)

    print(f"\nSaved summary metrics: {SUMMARY_METRICS_PATH}")
    print(f"Saved per-class metrics: {PER_CLASS_METRICS_PATH}")
    print(f"Saved confusion matrices: {CONFUSION_MATRIX_PATH}")

    if not summary_df.empty:
        print("\nSummary metrics:")
        print(summary_df)

    if not per_class_all_df.empty:
        print("\nPer-class metrics:")
        print(per_class_all_df)


# ============================================================
# Main experiment
# ============================================================
def main() -> None:
    ensure_output_dir()
    train_df, val_df = load_mahed_data()
    subset_df = load_or_create_subset(val_df)
    few_shot_examples = load_or_create_few_shot_examples(train_df)
    client = get_gemini_client()

    for prompt_type in PROMPT_TYPES:

        if prompt_type in ["few_shot_definition", "three_shot_cultural"]:
            examples = few_shot_examples
        else:
            examples = None

        run_prompt_experiment(
            client=client,
            subset_df=subset_df,
            prompt_type=prompt_type,
            few_shot_examples=examples,
            batch_size=BATCH_SIZE,
            sleep_sec=SLEEP_SEC,
        )

        save_all_metrics(PROMPT_TYPES)

    print("\nExperiment finished or paused. Re-run this file to resume unfinished rows.")


if __name__ == "__main__":
    main()


Train distribution:
label
not_applicable    3697
hope              1892
hate              1301
Name: count, dtype: int64

Validation distribution:
label
not_applicable    806
hope              409
hate              261
Name: count, dtype: int64
Loaded existing subset: /content/drive/MyDrive/496/MAHED_Gemini_Task3/mahed_val_subset_500.csv

Subset distribution:
label
not_applicable    273
hope              139
hate               88
Name: count, dtype: int64
Loaded existing few-shot examples: /content/drive/MyDrive/496/MAHED_Gemini_Task3/few_shot_examples.csv
Loaded existing results for three_shot_cultural: /content/drive/MyDrive/496/MAHED_Gemini_Task3/results_three_shot_cultural_gemini_2_5_flash.csv

Prompt: three_shot_cultural
Remaining samples: 125 / 500
Request failed. Retrying in 10s. Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.go

KeyboardInterrupt: 